# GKG Pipeline Demo

**Flow:** clone repo -> AST map -> designer blueprint -> implementer diffs -> optional apply

1. Set `REPO_URL` and `FEATURE` in the config cell
2. Run all cells top-to-bottom
3. Review diffs before running the apply cell

In [14]:
import subprocess, shutil, sys, os, json, time, stat
from pathlib import Path
from IPython.display import display, HTML, Markdown

# -- path guard: works whether launched from GKG dir or anywhere else --
_here = Path(os.path.abspath('')).resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

from ollama_client import OllamaClient
from ast_mapper import map_project
from designer import design_feature
from implementer import implement_feature
from pipeline import run as pipeline_run, apply_changes, PipelineResult
from gkg_viz import render_html
from commands import describe_for_coder
from gkg import Level, MacroPayload, MesoPayload, MicroPayload, DesignPattern

# -- CONFIG ---------------------------------------------------------------
REPO_URL = 'https://github.com/MoonFlowww/Sepia'
FEATURE  = 'add scientific notation toggle for axis tick labels'
MODEL    = 'qwen2.5-coder:7b'
WORK_DIR = Path('_demo_repos')
FRESH    = True
# -------------------------------------------------------------------------

client = OllamaClient(MODEL)
try:
    pong = client.complete('reply with the single word: pong', max_tokens=10)
    print(f'ollama ok . model={MODEL!r}  reply={pong.strip()!r}')
except Exception as e:
    print(f'ollama not reachable: {e}\nStart ollama and re-run.')

ollama ok . model='qwen2.5-coder:7b'  reply='pong'


## 1 . Clone (fresh)

In [15]:
import os, stat

def _force_remove(func, path, exc_info):
    """onerror handler: clear read-only bit then retry (needed for .git on Windows)."""
    os.chmod(path, stat.S_IWRITE)
    func(path)

def clone_fresh(url: str, work_dir: Path, fresh: bool) -> Path:
    repo_name = url.rstrip('/').split('/')[-1].removesuffix('.git')
    dest = work_dir / repo_name
    if fresh and dest.exists():
        print(f'removing {dest} ...')
        shutil.rmtree(dest, onerror=_force_remove)
    if not dest.exists():
        work_dir.mkdir(parents=True, exist_ok=True)
        print(f'cloning {url} ...')
        result = subprocess.run(
            ['git', 'clone', '--depth=1', url, str(dest)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f'git clone failed:\n{result.stderr}')
        print(f'cloned -> {dest}')
    else:
        print(f'using existing clone: {dest}')
    return dest

repo_path = clone_fresh(REPO_URL, WORK_DIR, FRESH)
print(f'repo: {repo_path}')

removing _demo_repos\Sepia ...
cloning https://github.com/MoonFlowww/Sepia ...
cloned -> _demo_repos\Sepia
repo: _demo_repos\Sepia


## 2 . Run pipeline  (map -> design -> implement)

In [16]:
t0 = time.time()
result: PipelineResult = pipeline_run(str(repo_path), FEATURE, client)
print(f'\ntotal: {time.time()-t0:.1f}s')
print()
print(result.summary())
print()
stats = client.stats_summary()
print(f"tokens  -  prompt: {stats['prompt_tokens']}  completion: {stats['completion_tokens']}  total: {stats['total_tokens']}")

[1/3] mapping '_demo_repos\\Sepia'
      158 nodes, 380 edges
[2/3] designing 'add scientific notation toggle for axis tick labels'
      8 blueprint nodes
      Add a feature to toggle between normal and scientific notation for axis tick labels in plots.
[3/3] implementing
      6 file(s) with changes

total: 124.9s

nodes mapped:    158
edges mapped:    380
blueprint nodes: 8
files changed:   6
  ScientificNotationToggle.py: 13 diff lines
  new_module.py: 11 diff lines
  Figure.py: 13 diff lines
  plot_figure: 20 diff lines
  TickEngine.py: 8 diff lines
  rendering_core: 13 diff lines

tokens  -  prompt: 28891  completion: 3478  total: 32369


## 2b . Visualize GKG

In [17]:
viz_path = render_html(result.graph, "sepia_gkg.html")
abs_path = os.path.abspath(viz_path)
display(HTML(
    f'<p style="font-family:monospace;font-size:12px">'
    f'Graph -> <a href="file:///{abs_path.replace(chr(92), "/")}" target="_blank">{abs_path}</a>'
    f'<br><small style="color:#888">open in Chrome/Edge . nodes={len(result.graph.nodes)} edges={len(result.graph.edges)}</small></p>'
))

edge_kinds = {}
for e in result.graph.edges.values():
    edge_kinds[e.kind.value] = edge_kinds.get(e.kind.value, 0) + 1
for lvl in ("MACRO", "MESO", "MICRO"):
    c = sum(1 for n in result.graph.nodes.values() if n.level.value == lvl)
    print(f"  {lvl:<6}: {c} nodes")
for k, v in sorted(edge_kinds.items()):
    print(f"  {k:<14}: {v} edges")

  MACRO : 10 nodes
  MESO  : 26 nodes
  MICRO : 122 nodes
  CALLS         : 139 edges
  DEPENDS_ON    : 14 edges
  OWN           : 148 edges
  SEND          : 79 edges


## 7 . Reset repo  *(optional  -  restores clone to HEAD)*

In [18]:
subprocess.run(['git', '-C', str(repo_path), 'checkout', '--', '.'], check=True)
subprocess.run(['git', '-C', str(repo_path), 'clean', '-fd'], check=True)
print('repo reset to HEAD')

repo reset to HEAD


---
## A/B Quest Runner

Three conditions per quest:
| Condition | Context given | Turns | What it tests |
|-----------|--------------|-------|---------------|
| **baseline_full** | Entire codebase dumped upfront | 1 | Raw LLM without navigation |
| **baseline_nav** | File tree only, LLM reads files | N | Blind file navigation |
| **gkg_nav** | GKG MACRO map + auto-route hint | N | Map-guided navigation |


In [19]:
from gkg_navigator import GKGNavigator
from ab_quests import QUESTS
from ab_runner import run_baseline_full, run_baseline_nav, run_gkg_nav, run_quest_ab

from ast_mapper import map_project as _map
_g = _map(str(repo_path))
navigator = GKGNavigator(_g, str(repo_path))
print("Navigator ready.")
print(navigator.dump())

Navigator ready.
=== MACRO LEVEL ===
  core                  Module core                                              [5 classes]
  data                  Module data                                              [4 classes]
  params                Module params                                            [7 classes]
  plot2d                Module plot2d                                            [4 classes]
  rendering             Module rendering                                         [5 classes]
  stress                Module stress                                            [1 classes]


### Run a single quest

| id | name | complexity |
|----|------|------------|
| 0 | Find & Repeat Function | retrieval |
| 1 | Scientific Notation on Axis | local_add |
| 2 | Line/Scatter Mode Switch | cross_module |
| 3 | Regression Option on Plot | new_feature |
| 4 | Histogram Plot | new_feature |
| 5 | Low-Latency Optimisation Audit | analysis |

In [20]:
QUEST_ID = 1  # 0..5
quest = QUESTS[QUEST_ID]
print("Quest:", quest.id, quest.name, "(", quest.complexity, ")")
print()
print("Prompt:", quest.prompt)
print("Target:", quest.target_node)
print("Criteria:", quest.success_criteria)

Quest: 1 Scientific Notation on Axis ( local_add )

Prompt: Add a method `set_scientific_notation(bool enabled, int precision=2)` to the axis tick formatting logic. When enabled, labels should be formatted as e.g. '1.23e+04' instead of '12300'. Show the modified class.
Target: TickEngine
Criteria: New method added to the correct class. Uses snprintf/std::format or equivalent for scientific notation. No unrelated changes.


#### baseline_full  (entire codebase, 1 shot)

In [21]:
print("=== BASELINE_FULL Q" + str(quest.id) + " ===")
bf = run_baseline_full(quest, client, str(repo_path))
print("turns:", bf.turns, "  tokens:", bf.total_tokens)
print()
print("Answer:")
print(bf.notes)

=== BASELINE_FULL Q1 ===
turns: 1   tokens: 13375

Answer:
```cpp
// === include/sepia/rendering/tick_engine.hpp ===
#pragma once

#include "../core/types.hpp"
#include <vector>
#include <cmath>
#include <string>
#include <cstdio>

namespace sepia{ namespace rendering {

struct Tick {
  f64         value;
  std::string label;
  bool        is_minor = false;
};

class TickEngine {
public:
  static std::vector<Tick> compute(f64 lo, f64 hi, usize target_ticks = 6,
                                   bool include_minor = false,
                                   ScaleType scale = ScaleType::Linear) {
    if (scale == ScaleType::Log)
      return compute_log(lo, hi, include_minor);
    return compute_linear(lo, hi, target_ticks, include_minor);
  }

private:
  static std::vector<Tick> compute_linear(f64 lo, f64 hi, usize target_ticks, bool include_minor) {
    std::vector<Tick> ticks;
    if (hi <= lo) return ticks;

    f64 range = hi - lo;
    f64 rough_step = range / static_cast<f64>(targ

#### baseline_nav  (file tree, navigate by LIST/READ)

In [22]:
print("=== BASELINE_NAV Q" + str(quest.id) + " ===")
bn = run_baseline_nav(quest, client, str(repo_path))
print("turns:", bn.turns, "  tokens:", bn.total_tokens, "  files:", bn.files_opened)
print()
print("Answer:")
print(bn.notes)

=== BASELINE_NAV Q1 ===
turns: 3   tokens: 2888   files: 1

Answer:
```cpp
#pragma once

#include "../core/types.hpp"
#include <vector>
#include <cmath>
#include <string>
#include <cstdio>

namespace sepia{ namespace rendering {

struct Tick {
  f64         value;
  std::string label;
  bool        is_minor = false;
};

class TickEngine {
public:
  static std::vector<Tick> compute(f64 lo, f64 hi, usize target_ticks = 6,
                                   bool include_minor = false,
                                   ScaleType scale = ScaleType::Linear) {
    if (scale == ScaleType::Log)
      return compute_log(lo, hi, include_minor);
    return compute_linear(lo, hi, target_ticks, include_minor);
  }

  static void set_scientific_notation(bool enabled, int precision=2) {
    scientific_notation_enabled = enabled;
    scientific_notation_precision = precision;
  }

private:
  static std::vector<Tick> compute_linear(f64 lo, f64 hi, usize target_ticks, bool include_minor) {
    std::vect

#### gkg_nav  (GKG map + auto-route, navigate by CD/GET_CODE)

In [23]:
print("=== GKG_NAV Q" + str(quest.id) + " ===")
gn = run_gkg_nav(quest, client, navigator)
print("turns:", gn.turns, "  tokens:", gn.total_tokens, "  fetches:", gn.files_opened)
print()
print("Answer:")
print(gn.notes)

=== GKG_NAV Q1 ===
turns: 4   tokens: 3044   fetches: 1

Answer:
```cpp
static std::vector<Tick> compute(f64 lo, f64 hi, usize target_ticks = 6,
                                   bool include_minor = false,
                                   ScaleType scale = ScaleType::Linear) {
    if (scale == ScaleType::Log)
      return compute_log(lo, hi, include_minor);
    return compute_linear(lo, hi, target_ticks, include_minor);
  }

void set_scientific_notation(bool enabled, int precision = 2) {
    // Implementation of set_scientific_notation
}
```


#### Delta table

In [24]:
def show_conversation(metrics, title, max_content=600):
    from IPython.display import display, HTML
    NL = chr(10)
    role_style = {
        'assistant': 'background:#1a2a3a;color:#7eafc8;border-left:3px solid #4a90d9',
        'tool':      'background:#1a2a1a;color:#7ec87e;border-left:3px solid #4aaa4a',
    }
    rows = []
    rows.append('<h4 style="margin:8px 0 4px;color:#ccc">' + title + '</h4>')
    rows.append('<div style="font-family:monospace;font-size:11px">')
    conv = getattr(metrics, 'conversation', [])
    if not conv:
        rows.append('<div style="color:#888">(no log - re-run quest cells)</div>')
    for entry in conv:
        role    = entry.get('role', '?')
        content = entry.get('content', '')
        cmd     = entry.get('cmd', '')
        st = role_style.get(role, 'background:#2a2a2a;color:#ccc')
        if role == 'assistant':
            # show only the parsed command badge, not the raw content (they are the same)
            badge = '<span style="background:#333;padding:1px 5px;border-radius:3px">' + cmd + '</span>'
            rows.append('<div style="' + st + ';padding:4px 10px;margin:2px 0;border-radius:4px"><b>LLM</b> ' + badge + '</div>')
        else:
            if len(content) > max_content:
                content = content[:max_content] + '... [truncated]'
            esc = (content
                   .replace('&', '&amp;')
                   .replace('<', '&lt;')
                   .replace('>', '&gt;')
                   .replace(NL, '<br>'))
            rows.append('<div style="' + st + ';padding:4px 10px;margin:2px 0;border-radius:4px"><b>TOOL</b><br>' + esc + '</div>')
    rows.append('</div>')
    display(HTML(''.join(rows)))

show_conversation(bn, 'baseline_nav (LIST/READ)  turns=' + str(bn.turns) + '  tokens=' + str(bn.total_tokens))
show_conversation(gn, 'gkg_nav (CD/LIST_FILE/GET_CODE)  turns=' + str(gn.turns) + '  tokens=' + str(gn.total_tokens))


### Run all 6 quests - full A/B report

> Warning: ~20 min with a 1.5b model (3 conditions x 6 quests).

In [25]:
all_results = []
for q in QUESTS:
    print("")
    print("--- Q{}: {} ---".format(q.id, q.name))
    bf, bn, gn = run_quest_ab(q, client, str(repo_path), navigator)
    all_results.append((bf, bn, gn))
print("")
print("Done.")


--- Q0: Find & Repeat Function ---
  [full]  Q0: Find & Repeat Function ...
    turns=1 tokens=12209 f1=44%
  [nav]   Q0: Find & Repeat Function ...
    turns=3 tokens=2799 f1=59%
  [gkg]   Q0: Find & Repeat Function ...
    turns=4 tokens=2421 f1=98%

--- Q1: Scientific Notation on Axis ---
  [full]  Q1: Scientific Notation on Axis ...
    turns=1 tokens=13375 f1=25%
  [nav]   Q1: Scientific Notation on Axis ...
    turns=3 tokens=2888 f1=75%
  [gkg]   Q1: Scientific Notation on Axis ...
    turns=4 tokens=3044 f1=75%

--- Q2: Line/Scatter Mode Switch ---
  [full]  Q2: Line/Scatter Mode Switch ...
    turns=1 tokens=12469 f1=100%
  [nav]   Q2: Line/Scatter Mode Switch ...
    turns=3 tokens=5552 f1=100%
  [gkg]   Q2: Line/Scatter Mode Switch ...
    turns=4 tokens=2784 f1=100%

--- Q3: Regression Option on Plot ---
  [full]  Q3: Regression Option on Plot ...
    turns=1 tokens=13379 f1=100%
  [nav]   Q3: Regression Option on Plot ...
    turns=3 tokens=5646 f1=100%
  [gkg]   Q3: Regr

In [26]:
# Optional: run LLM judge (gemma4:26b) on all results
# Requires: ollama pull gemma4:26b
from ab_runner import JUDGE_MODEL, score_with_judge
from ollama_client import OllamaClient

judge_client = OllamaClient(JUDGE_MODEL)
for (bf, bn, gn), q in zip(all_results, QUESTS):
    print('judging Q{} ...'.format(q.id), end=' ', flush=True)
    score_with_judge(bf, q, judge_client)
    score_with_judge(bn, q, judge_client)
    score_with_judge(gn, q, judge_client)
    print('full={:.2f}  nav={:.2f}  gkg={:.2f}'.format(
        max(bf.judge_score,0), max(bn.judge_score,0), max(gn.judge_score,0)))
print('Done. Re-run the table cell above to see judge scores.')


judging Q0 ... full=0.00  nav=0.00  gkg=0.00
judging Q1 ... full=0.00  nav=0.00  gkg=0.00
judging Q2 ... full=0.00  nav=0.00  gkg=0.00
judging Q3 ... full=0.00  nav=0.00  gkg=0.00
judging Q4 ... full=0.00  nav=0.00  gkg=0.00
judging Q5 ... full=0.00  nav=0.00  gkg=0.00
Done. Re-run the table cell above to see judge scores.


In [27]:
from ab_runner import score_metrics
from ab_stats import quality_table, significance_caveat

# re-score with latest score_metrics (picks up judge scores if run)
for (bf, bn, gn), q in zip(all_results, QUESTS):
    score_metrics(bf, q, navigator)
    score_metrics(bn, q, navigator)
    score_metrics(gn, q, navigator)

print(quality_table(all_results, QUESTS))


AttributeError: 'RunMetrics' object has no attribute 'quality'